In [0]:
#### Below code it to cross check numbers are correct or not ####

item_counts = {}

# Count each item
for sublist in transactions:
    for item in sublist:
        if item in item_counts:
            item_counts[item] += 1
        else:
            item_counts[item] = 1

# Task 1

In [0]:
import requests

url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv"

content = requests.get(url).text

rows = content.strip().split("\n")

transactions = []
for row in rows:
    transaction = []
    items = row.split(",")
    for item in items:
        transaction.append(item.strip())
    transactions.append(transaction)

###### the same thing can be done using list comprehension also as shown below #######

# transactions = [
#     [item.strip() for item in row.split(",")]
#     for row in content.strip().split("\n")
#     ]

# Converting to DataFrame instead of RDD (RDDs not supported on Serverless databricks free  edition)
from pyspark.sql import Row

# Creating DataFrame from transactions list
groceries_df = spark.createDataFrame([Row(items=transaction) for transaction in transactions])




# Task 2

In [0]:

#### Code to write to text file for all the unique products  ####

from pyspark.sql.functions import explode

unique_products = (
    groceries_df
    .select(explode("items").alias("product"))
    .distinct()
    .orderBy("product")
)

content_product = "\n".join(
    row["product"]
    for row in unique_products.collect()
)

dbutils.fs.put(
    "dbfs:/Workspace/test-assignment/out/out_1_2a.txt",
    content_product,
    overwrite=True
)

# If Using RDD was supported in Databricks free edition below code would work

# Create RDD
# groceries_rdd = spark.sparkContext.parallelize(transactions)

# Get unique products
# unique_products_rdd = (
#     groceries_rdd
#     .flatMap(lambda transaction: transaction)
#     .distinct()
#     .sortBy(lambda product: product)
# )

print("Text file created successfully")

Wrote 2032 bytes.
Text file created successfully


In [0]:
#### Code to write to text file for all the count of unique products  ####


count_value = unique_products.count()
dbutils.fs.put(
    "dbfs:/Workspace/test-assignment/out/out_1_2b.txt",
    f"count: {count_value}",
    overwrite=True
)

Wrote 10 bytes.


True

# Task 3

In [0]:
#### Code to write to text file for top 5 purchase products  ####

from pyspark.sql.functions import explode, col, desc

top5_products_df = (
    groceries_df
    .select(explode(col("items")).alias("product"))
    .groupBy("product")
    .count()
    .orderBy(desc("count"))
    .limit(5)
)

top5_content = "\n".join(
    f"('{row['product']}', {row['count']})"
    for row in top5_products_df.collect()
)

dbutils.fs.put(
    "dbfs:/Workspace/test-assignment/out/out_1_3.txt",
    top5_content,
    overwrite=True
)


# If Using RDD was supported in Databricks free edition below code would work

# groceries_rdd = spark.sparkContext.parallelize(transactions)

# top5_products = (
#     groceries_rdd
#     .flatMap(lambda transaction: transaction)
#     .map(lambda product: (product, 1))
#     .reduceByKey(lambda x, y: x + y)
#     .sortBy(lambda x: x[1], ascending=False)
#     .take(5)
# )

Wrote 100 bytes.


True